In [68]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots


## Initialize, create and clean initial data frame

In [69]:
data_root = "/Users/the87/Downloads/IMC_GoTritons/round_1/data"

In [70]:
price_d0 = pd.read_csv(f"{data_root}/prices_round_1_day_0.csv",  sep=";")
trade_d0 = pd.read_csv(f"{data_root}/trades_round_1_day_0.csv",  sep=";"); trade_d0["day"] = 0
price_d1 = pd.read_csv(f"{data_root}/prices_round_1_day_-1.csv", sep=";")
trade_d1 = pd.read_csv(f"{data_root}/trades_round_1_day_-1.csv", sep=";"); trade_d1["day"] = -1
price_d2 = pd.read_csv(f"{data_root}/prices_round_1_day_-2.csv", sep=";")
trade_d2 = pd.read_csv(f"{data_root}/trades_round_1_day_-2.csv", sep=";"); trade_d2["day"] = -2

print("Products:", sorted(pd.concat([price_d0, price_d1, price_d2])["product"].unique()))


Products: ['ASH_COATED_OSMIUM', 'INTARIAN_PEPPER_ROOT']


In [71]:
# Drop columns where ALL values are NaN
price_d0 = price_d0.dropna(axis=1, how="all")
trade_d0 = trade_d0.dropna(axis=1, how="all")
price_d1 = price_d1.dropna(axis=1, how="all")
trade_d1 = trade_d1.dropna(axis=1, how="all")
price_d2 = price_d2.dropna(axis=1, how="all")
trade_d2 = trade_d2.dropna(axis=1, how="all")

# Drop currency column from trades (all values are the same)
trade_d0 = trade_d0.drop(columns=["currency"])
trade_d1 = trade_d1.drop(columns=["currency"])
trade_d2 = trade_d2.drop(columns=["currency"])

print("price_d0 columns:", price_d0.columns.tolist())
print("trade_d0 columns:", trade_d0.columns.tolist())


price_d0 columns: ['day', 'timestamp', 'product', 'bid_price_1', 'bid_volume_1', 'bid_price_2', 'bid_volume_2', 'bid_price_3', 'bid_volume_3', 'ask_price_1', 'ask_volume_1', 'ask_price_2', 'ask_volume_2', 'ask_price_3', 'ask_volume_3', 'mid_price', 'profit_and_loss']
trade_d0 columns: ['timestamp', 'symbol', 'price', 'quantity', 'day']


## Create data frame per product and data frame per day

In [72]:

# ── DataFrames by product ─────────────────────────────────────────────────────
# Combine all days, then filter by product (prices and trades separately)
all_prices = pd.concat([price_d0, price_d1, price_d2], ignore_index=True)
all_trades = pd.concat([trade_d0, trade_d1, trade_d2], ignore_index=True)

prices_pepper  = all_prices[all_prices["product"] == "INTARIAN_PEPPER_ROOT"].reset_index(drop=True)
prices_osmium  = all_prices[all_prices["product"] == "ASH_COATED_OSMIUM"].reset_index(drop=True)

trades_pepper  = all_trades[all_trades["symbol"] == "INTARIAN_PEPPER_ROOT"].reset_index(drop=True)
trades_osmium  = all_trades[all_trades["symbol"] == "ASH_COATED_OSMIUM"].reset_index(drop=True)

print("INTARIAN_PEPPER_ROOT prices:", len(prices_pepper), "| trades:", len(trades_pepper))
print("ASH_COATED_OSMIUM    prices:", len(prices_osmium), "| trades:", len(trades_osmium))

# ── DataFrames by day ─────────────────────────────────────────────────────────
# Each day df merges prices + trades for that day
def make_day_df(price_df, trade_df):
    prices = price_df.copy()
    trades = trade_df.copy().rename(columns={"symbol": "product"})
    trades["record_type"] = "trade"
    prices["record_type"] = "price"
    return pd.concat([prices, trades], ignore_index=True).sort_values("timestamp").reset_index(drop=True)

day_0 = make_day_df(price_d0, trade_d0)
day_1 = make_day_df(price_d1, trade_d1)
day_2 = make_day_df(price_d2, trade_d2)

print("\nDay  0:", len(day_0), "rows")
print("Day -1:", len(day_1), "rows")
print("Day -2:", len(day_2), "rows")


INTARIAN_PEPPER_ROOT prices: 30000 | trades: 1011
ASH_COATED_OSMIUM    prices: 30000 | trades: 1265

Day  0: 20743 rows
Day -1: 20760 rows
Day -2: 20773 rows


## Price Graph, pre product per day

In [73]:

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Mid price plots (from price dataframes) ───────────────────────────────────
fig = make_subplots(rows=2, cols=1,
                    subplot_titles=["INTARIAN_PEPPER_ROOT — Mid Price",
                                    "ASH_COATED_OSMIUM — Mid Price"],
                    shared_xaxes=False, vertical_spacing=0.12)

day_styles = {-2: ("steelblue", "Day -2"), -1: ("darkorange", "Day -1"), 0: ("green", "Day 0")}

for row, (prices, name) in enumerate([(prices_pepper, "INTARIAN_PEPPER_ROOT"),
                                       (prices_osmium,  "ASH_COATED_OSMIUM")], start=1):
    for day_val, (color, label) in day_styles.items():
        subset = prices[prices["day"] == day_val]
        fig.add_trace(
            go.Scatter(x=subset["timestamp"], y=subset["mid_price"],
                       mode="lines", name=label, legendgroup=label,
                       showlegend=(row == 1),
                       line=dict(color=color, width=1)),
            row=row, col=1
        )

fig.update_layout(height=700, title="Mid Price by Product", hovermode="x unified")
fig.update_xaxes(title_text="Timestamp")
fig.update_yaxes(title_text="Mid Price")
fig.show()


## Trade Graph, per product per day

In [74]:

# ── Trade price plots (from trade dataframes) ─────────────────────────────────
fig2 = make_subplots(rows=2, cols=1,
                     subplot_titles=["INTARIAN_PEPPER_ROOT — Trade Prices",
                                     "ASH_COATED_OSMIUM — Trade Prices"],
                     shared_xaxes=False, vertical_spacing=0.12)

for row, (trades, name) in enumerate([(trades_pepper, "INTARIAN_PEPPER_ROOT"),
                                       (trades_osmium,  "ASH_COATED_OSMIUM")], start=1):
    for day_val, (color, label) in day_styles.items():
        subset = trades[trades["day"] == day_val]
        fig2.add_trace(
            go.Scatter(x=subset["timestamp"], y=subset["price"],
                       mode="markers", name=label, legendgroup=label,
                       showlegend=(row == 1),
                       marker=dict(color=color, size=5, opacity=0.7),
                       customdata=subset[["quantity"]],
                       hovertemplate="Timestamp: %{x}<br>Price: %{y}<br>Qty: %{customdata[0]}<extra>" + label + "</extra>"),
            row=row, col=1
        )

fig2.update_layout(height=700, title="Trade Prices by Product", hovermode="closest")
fig2.update_xaxes(title_text="Timestamp")
fig2.update_yaxes(title_text="Trade Price")
fig2.show()


## Price Graph, per product, contineous time

In [75]:

# ── Continuous mid price plots (day -2 → day -1 → day 0) ─────────────────────
# Timestamps reset to 0 each day, so offset them to make x-axis continuous
def add_continuous_ts(df, day_col='day'):
    df = df.copy().sort_values([day_col, 'timestamp'])
    max_ts = df.groupby(day_col)['timestamp'].max()
    offset = 0
    offsets = {}
    for day in sorted(df[day_col].unique()):  # -2, -1, 0
        offsets[day] = offset
        offset += max_ts[day] + 100  # small gap between days
    df['cont_ts'] = df.apply(lambda r: r['timestamp'] + offsets[r[day_col]], axis=1)
    return df

prices_pepper_c = add_continuous_ts(prices_pepper)
prices_osmium_c = add_continuous_ts(prices_osmium)

fig3 = make_subplots(rows=2, cols=1,
                     subplot_titles=['INTARIAN_PEPPER_ROOT - Continuous Mid Price',
                                     'ASH_COATED_OSMIUM - Continuous Mid Price'],
                     vertical_spacing=0.12)

for row, (prices_c, name) in enumerate([(prices_pepper_c, 'INTARIAN_PEPPER_ROOT'),
                                         (prices_osmium_c,  'ASH_COATED_OSMIUM')], start=1):
    for day_val, (color, label) in day_styles.items():
        subset = prices_c[prices_c['day'] == day_val]
        fig3.add_trace(
            go.Scatter(x=subset['cont_ts'], y=subset['mid_price'],
                       mode='lines', name=label, legendgroup=label,
                       showlegend=(row == 1),
                       line=dict(color=color, width=1),
                       customdata=subset['timestamp'],
                       hovertemplate=f'Day: {day_val}<br>Timestamp: %{{customdata}}<br>Price: %{{y}}<extra></extra>'),
            row=row, col=1
        )

fig3.update_layout(height=700, title='Continuous Mid Price by Product (Day -2 to Day 0)',
                   hovermode='x unified')
fig3.update_xaxes(title_text='Continuous Timestamp')
fig3.update_yaxes(title_text='Mid Price')
fig3.show()


## Trade Graph, per product, contineous day

In [76]:

# ── Continuous trade prices + per-day box plots ───────────────────────────────
fig4 = make_subplots(rows=2, cols=1,
                     subplot_titles=['INTARIAN_PEPPER_ROOT - Continuous Trade Prices + Box Plot',
                                     'ASH_COATED_OSMIUM - Continuous Trade Prices + Box Plot'],
                     vertical_spacing=0.12)

for row, (trades_c, name) in enumerate([
    (trades_pepper_c, 'INTARIAN_PEPPER_ROOT'),
    (trades_osmium_c, 'ASH_COATED_OSMIUM'),
], start=1):
    for day_val, (color, label) in day_styles.items():
        subset = trades_c[trades_c['day'] == day_val]
        x_min = subset['cont_ts'].min()
        x_max = subset['cont_ts'].max()
        x_mid = (x_min + x_max) / 2
        x_range = x_max - x_min

        # Scatter: individual trades
        fig4.add_trace(
            go.Scatter(x=subset['cont_ts'], y=subset['price'],
                       mode='markers', name=label, legendgroup=label,
                       showlegend=(row == 1),
                       marker=dict(color=color, size=4, opacity=0.4),
                       customdata=subset[['timestamp', 'quantity']].values,
                       hovertemplate=f'Day: {day_val}<br>Timestamp: %{{customdata[0]}}<br>Price: %{{y}}<br>Qty: %{{customdata[1]}}<extra></extra>'),
            row=row, col=1
        )

        # One box plot per day: x is a constant (midpoint), width spans 60% of day
        fig4.add_trace(
            go.Box(x=[x_mid] * len(subset), y=subset['price'],
                   name=f'{label} box', legendgroup=f'{label}_box',
                   showlegend=(row == 1),
                   marker_color=color, line_color=color,
                   fillcolor=color, opacity=0.4,
                   boxmean=False,
                   width=x_range * 0.6,
                   hovertemplate=f'Day: {day_val}<br>Price: %{{y}}<extra>{label} box</extra>'),
            row=row, col=1
        )

fig4.update_layout(height=750,
                   title='Continuous Trade Prices by Product with Box Plot (Day -2 to Day 0)',
                   hovermode='closest', boxmode='overlay')
fig4.update_xaxes(title_text='Continuous Timestamp')
fig4.update_yaxes(title_text='Trade Price')
fig4.show()


In [78]:

# ── Trade prices + box plot + rolling 25th/75th percentile lines ──────────────
# At each trade timestamp, compute the 25th/75th percentile of all trades
# seen so far that day (cumulative expanding window within each day).

def add_rolling_percentiles(trades_c):
    """Add p10 and p90 columns: cumulative percentile within each (day, product)."""
    trades_c = trades_c.sort_values(['day', 'cont_ts']).copy()
    p10, p90 = [], []
    for (day_val,), group in trades_c.groupby(['day'], sort=True):
        prices_so_far = []
        for price in group['price']:
            prices_so_far.append(price)
            p10.append(pd.Series(prices_so_far).quantile(0.10))
            p90.append(pd.Series(prices_so_far).quantile(0.90))
    trades_c['p10'] = p10
    trades_c['p90'] = p90
    return trades_c

trades_pepper_p = add_rolling_percentiles(trades_pepper_c)
trades_osmium_p  = add_rolling_percentiles(trades_osmium_c)

fig5 = make_subplots(rows=2, cols=1,
                     subplot_titles=['INTARIAN_PEPPER_ROOT - Trades + Box + Rolling Percentiles',
                                     'ASH_COATED_OSMIUM - Trades + Box + Rolling Percentiles'],
                     vertical_spacing=0.12)

for row, (trades_p, name) in enumerate([
    (trades_pepper_p, 'INTARIAN_PEPPER_ROOT'),
    (trades_osmium_p,  'ASH_COATED_OSMIUM'),
], start=1):
    for day_val, (color, label) in day_styles.items():
        subset = trades_p[trades_p['day'] == day_val]
        x_min = subset['cont_ts'].min()
        x_max = subset['cont_ts'].max()
        x_mid = (x_min + x_max) / 2
        x_range = x_max - x_min

        # Scatter: individual trades
        fig5.add_trace(
            go.Scatter(x=subset['cont_ts'], y=subset['price'],
                       mode='markers', name=label, legendgroup=label,
                       showlegend=(row == 1),
                       marker=dict(color=color, size=4, opacity=0.4),
                       customdata=subset[['timestamp', 'quantity']].values,
                       hovertemplate=f'Day: {day_val}<br>Timestamp: %{{customdata[0]}}<br>Price: %{{y}}<br>Qty: %{{customdata[1]}}<extra></extra>'),
            row=row, col=1
        )

        # One box plot per day
        fig5.add_trace(
            go.Box(x=[x_mid] * len(subset), y=subset['price'],
                   name=f'{label} box', legendgroup=f'{label}_box',
                   showlegend=(row == 1),
                   marker_color=color, line_color=color,
                   fillcolor=color, opacity=0.4,
                   boxmean=False, width=x_range * 0.6,
                   hovertemplate=f'Day: {day_val}<br>Price: %{{y}}<extra>{label} box</extra>'),
            row=row, col=1
        )

        # Rolling 90th percentile line
        fig5.add_trace(
            go.Scatter(x=subset['cont_ts'], y=subset['p90'],
                       mode='lines', name=f'{label} p90', legendgroup=f'{label}_p90',
                       showlegend=(row == 1),
                       line=dict(color=color, width=1.5, dash='dot'),
                       customdata=subset['timestamp'],
                       hovertemplate=f'Day: {day_val}<br>Timestamp: %{{customdata}}<br>p90: %{{y:.2f}}<extra></extra>'),
            row=row, col=1
        )

        # Rolling 10th percentile line
        fig5.add_trace(
            go.Scatter(x=subset['cont_ts'], y=subset['p10'],
                       mode='lines', name=f'{label} p10', legendgroup=f'{label}_p10',
                       showlegend=(row == 1),
                       line=dict(color=color, width=1.5, dash='dash'),
                       customdata=subset['timestamp'],
                       hovertemplate=f'Day: {day_val}<br>Timestamp: %{{customdata}}<br>p10: %{{y:.2f}}<extra></extra>'),
            row=row, col=1
        )

fig5.update_layout(height=750,
                   title='Trade Prices + Box Plot + Rolling 10th/90th Percentile (Day -2 to Day 0)',
                   hovermode='closest', boxmode='overlay')
fig5.update_xaxes(title_text='Continuous Timestamp')
fig5.update_yaxes(title_text='Trade Price')
fig5.show()
